Connected to so_conda_test (Python 3.11.15)

In [141]:
import os
import argparse
import pandas as pd
import copy

In [8]:
ur_path = '/projects/splitorfs/work/split-orf-prediction/Output/run_07.04.2026-16.10.51_RI_contamination_subtraction/Unique_DNA_Regions_genomic_final.bed'
so_results = '/projects/splitorfs/work/split-orf-prediction/Output/run_07.04.2026-16.10.51_RI_contamination_subtraction/UniqueProteinORFPairs.txt'

In [ ]:
def load_so_results(so_results):
    predicted_so_orfs = pd.read_csv(so_results, header=0, sep='\t')
    so_transcripts = predicted_so_orfs['OrfTransID'].to_list()
    predicted_so_orfs['OrfPos'] = predicted_so_orfs['OrfPos'].apply(
        lambda x: x.split(','))
    predicted_so_orfs['OrfStarts'] = predicted_so_orfs['OrfPos'].apply(
        lambda x: [y.split('-')[0] for y in x])
    predicted_so_orfs['nr_SO_starts'] = predicted_so_orfs['OrfPos'].apply(
        lambda x: len(x))
    return predicted_so_orfs, so_transcripts


def explode_so_df(predicted_so_orfs):
    predicted_so_orfs = predicted_so_orfs[[
        'OrfTransID', 'OrfIDs', 'OrfStarts', 'geneID']].copy()
    predicted_so_orfs['OrfID'] = predicted_so_orfs.apply(
        lambda x: x['OrfIDs'].split(','), axis=1)
    predicted_so_orfs['OrfStart'] = predicted_so_orfs['OrfStarts']
    all_predicted_so_orfs = predicted_so_orfs.explode(
        ['OrfID', 'OrfStart'], ignore_index=True).copy()
    return all_predicted_so_orfs, predicted_so_orfs


def load_dna_ur_df(UR_path):
    dna_ur_df = pd.read_csv(UR_path, sep='\t', header=None, names=[
                            'chr', 'start', 'stop', 'ID', 'score', 'strand'])
    dna_ur_df['OrfID'] = dna_ur_df['ID'].str.split(
        ':').apply(lambda x: x[1])
    dna_ur_df['OrfTransID'] = dna_ur_df['ID'].str.split(
        ':').apply(lambda x: x[0])

    return dna_ur_df


def classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs):
    # group several exonic URs per ORF together, several URs per ORF are also grouped together
    dna_ur_df = dna_ur_df.groupby('OrfID').agg({'start': 'min',
                                                'stop': 'max',
                                                        'chr': 'first',
                                                        'ID': lambda x: ','.join(x),
                                                        'OrfTransID': 'first'}).reset_index().copy()

    # map ORF positions to IDs
    orf_id_position_map = all_predicted_so_orfs.set_index('OrfID')[
        'OrfPosition']
    dna_ur_df['OrfPosition'] = dna_ur_df['OrfID'].map(orf_id_position_map
                                                      )

    # concatenate genomic regions
    dna_ur_df['genomic_UR'] = dna_ur_df['chr'].astype(
        str) + '_' + dna_ur_df['start'].astype(str) + '_' + dna_ur_df['stop'].astype(str)

    dna_ur_df['OverlapPercentage'] = 0.0
    dna_ur_df['OrfPositionsOverlapping'] = dna_ur_df['OrfPosition'].apply(
        lambda x: set([x]))
    dna_ur_df['OrfIDsOverlapping'] = dna_ur_df['OrfID'].apply(
        lambda x: set([x]))
    return dna_ur_df


def calculate_overlapping_region_percentage(start1, end1, start2, end2):
    if end1 <= start2 or end2 <= start1:
        return 0
    elif start1 < end2 and start2 < end1:
        overlap_start = max(start2, start1)
        overlap_end = min(end1, end2)
        nr_bp_overlap = overlap_end - overlap_start
        shorter_region = min(end2-start2, end1-start1)
        return nr_bp_overlap/shorter_region


def get_max_overlap_of_regions_in_df(chr_df, threshold=0.2):
    starts = chr_df['start'].to_numpy()
    ends = chr_df['stop'].to_numpy()

    for i in range(len(starts)):
        for j in range(i + 1, len(starts)):  # avoid duplicate + self-comparison
            overlap = calculate_overlapping_region_percentage(
                starts[i], ends[i], starts[j], ends[j]
            )
            if overlap >= threshold:
                chr_df.loc[j, 'OrfPositionsOverlapping'] = chr_df.loc[j,
                                                                      'OrfPositionsOverlapping'] | {chr_df.loc[i, 'OrfPosition']}
                chr_df.loc[j, 'OrfIDsOverlapping'] | {chr_df.loc[i, 'OrfID']}
                chr_df.loc[i, 'OrfPositionsOverlapping'] | {
                    chr_df.loc[j, 'OrfPosition']}
                chr_df.loc[i, 'OrfIDsOverlapping'] | {chr_df.loc[j, 'OrfID']}
                if overlap >= float(chr_df.loc[i, 'OverlapPercentage']):
                    chr_df.loc[i, 'OverlapPercentage'] = overlap
                if overlap >= float(chr_df.loc[j, 'OverlapPercentage']):
                    chr_df.loc[j, 'OverlapPercentage'] = overlap
    return chr_df

def summarize_overlapping_urs(gene_df, col_index):
    '''
    check per gene for overlapping unique regions and only keep the first instance
    this is done because not always the same ORFs overlap for overlapping unique regions
    '''
    if len(gene_df.index) > 1:
        gene_df_return = gene_df.copy()
        # search for pairwise overlaps of the OrfIDsOverlapping
        for index1 in gene_df.index:
            # compare 0-1, 0-2, 0-3, 1-2, 1-3, 2-3
            index2 = index1
            while index2 < len(gene_df.index) - 1:
                index2 = index2 + 1
                orf_id_overlap_1 = gene_df.iloc[index1, col_index]
                orf_id_overlap_2 = gene_df.iloc[index2, col_index]
                # if ORF IDs do overlap
                if len(orf_id_overlap_1 & orf_id_overlap_2) > 0:
                    # check if index still exists or is already removed
                    if index2 in gene_df_return.index:
                        # always keep index1: ensure that one region of the overlapping
                        # ones is kept in the end!
                        gene_df_return = gene_df_return.drop(index=index2)
        return gene_df_return
    else:
        # return gene df if not several URs per gene
        return gene_df


def get_so_position_in_transcript(so_df):
    # sort the ORF starts by position
    so_df['OrfStarts'] = so_df.apply(
        lambda x: sorted([int(start) for start in x['OrfStarts']]), axis=1)
    # map the ORF start to the respective position in the sorted list
    # indicate whether it is the first or a later (first, middle, last)
    so_df['OrfIndex'] = so_df.apply(
        lambda x: x['OrfStarts'].index(int(x['OrfStart'])), axis=1)
    so_df['OrfPosition'] = so_df.apply(lambda x: 'first' if x['OrfIndex'] == 0 else (
        'last' if x['OrfIndex'] == len(x['OrfStarts'])-1 else 'middle'), axis=1)
    return so_df


def identify_middle_unique_regions(row):
    if len(row['OrfPosition']) > 3:
        middle_indices = [index for index, position in enumerate(
            row['OrfPosition']) if position == 'middle']
        return sum([row['hasUR'][index] for index in middle_indices])
    elif len(row['OrfPosition']) == 3:
        return int(row['hasUR'][row['OrfPosition'].index('middle')])
    else:
        return 0


def format_categorization_df(so_categorization_df):
    def get_list_cols(so_categorization_df):
        list_cols = []
        for col in so_categorization_df.columns:
            if so_categorization_df[col].apply(lambda x: isinstance(x, list)).all():
                list_cols.append(col)
        return list_cols

    list_cols = get_list_cols(so_categorization_df)
    for col in list_cols:
        so_categorization_df[col] = so_categorization_df[col].apply(
            lambda x: ','.join(map(str, x)))
    return so_categorization_df


def so_transcript_categorization(dna_overlapping_ur_df, all_predicted_so_orfs):
    '''
    categorize Split-ORF transcripts by number of unique regions and whether these
    are in the first, middle or last ORF. Also give information about distinct unique 
    regions per transcript (multiple transcript isoforms are counted multiple times).
    Write CSV file of the results.
    '''

    all_predicted_so_orfs['hasUR'] = all_predicted_so_orfs['OrfID'].isin(
        dna_overlapping_ur_df['OrfID'])

    genomic_ur_dict = dict(
        zip(dna_overlapping_ur_df['OrfID'], dna_overlapping_ur_df['genomic_UR']))

    all_predicted_so_orfs['genomic_UR'] = all_predicted_so_orfs['OrfID'].map(
        genomic_ur_dict)

    # aggregating together conserves teh order!
    so_categorization_df = all_predicted_so_orfs.groupby('OrfTransID').agg({
        'OrfID': list,
        'OrfStart': list,
        'geneID': 'first',
        'OrfPosition': list,
        'genomic_UR': list,
        'hasUR': list}).reset_index().copy()

    so_categorization_df['nrOrfs'] = so_categorization_df['OrfID'].apply(
        lambda x: len(x))
    so_categorization_df['nrOrfsWithUR'] = so_categorization_df['hasUR'].apply(
        lambda x: sum(x))

    so_categorization_df['URInFirstORF'] = so_categorization_df.apply(
        lambda x: int(x.loc['hasUR'][x['OrfPosition'].index('first')]), axis=1)
    so_categorization_df['URInLastORF'] = so_categorization_df.apply(
        lambda x: int(x.loc['hasUR'][x['OrfPosition'].index('last')]), axis=1)

    so_categorization_df['URInMiddleORF'] = so_categorization_df.apply(
        lambda x: identify_middle_unique_regions(x), axis=1)

    # ur == ur filters out nn values
    # so_categorization_df['NrDistinctURs'] = so_categorization_df['genomic_UR'].apply(
    #     lambda x: len(set(ur for ur in x if ur == ur)))

    return so_categorization_df, all_predicted_so_orfs

In [150]:

def overlapping_ur_df_by_id(dna_ur_df, outdir, id, agg_col_index):
    '''
    group URs by gene, transcript or chromosomes and summarize if they overlap more than
    the indicated threshold: 0.2
    '''
    # get completely overlapping URs
    gene_dfs = {gene: copy.deepcopy(gene_df.reset_index(
        drop=True)) for gene, gene_df in dna_ur_df.groupby(id)}
    gene_dfs = {gene: get_max_overlap_of_regions_in_df(
        gene_df, 0.2) for gene, gene_df in gene_dfs.items()}
    dna_overlapping_ur_df = copy.deepcopy(pd.concat(
        gene_dfs.values()).reset_index(drop=True))

    dna_overlapping_ur_df['ORFs_sharing_region'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: len(x))
    dna_overlapping_ur_df['shared_region_type'] = dna_overlapping_ur_df['OrfPositionsOverlapping'].apply(
        lambda x: len(x))
    # frozenset: order within the set does not matter!
    dna_overlapping_ur_df.loc[:, 'OrfIDsOverlapping'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: frozenset(x))

    if id == 'OrfTransID':
        # aggregate ORFs that have overlapping URs with the exact same set of ORFs
        dna_distinct_ur_df = copy.deepcopy(dna_overlapping_ur_df.groupby('OrfIDsOverlapping').agg(
            {'genomic_UR': 'first',
             'ORFs_sharing_region': 'first',
             'OrfPosition': 'first',
             'ID': lambda x: ','.join(x),
             'OrfTransID': 'first',
             'OrfPositionsOverlapping': 'first',
             'OrfIDsOverlapping': 'first',
             'OverlapPercentage': 'max',
             'geneID': 'first',
             }).reset_index(drop=True))
    else:
        # aggregate ORFs that have overlapping URs with the exact same set of ORFs
        dna_distinct_ur_df = copy.deepcopy(dna_overlapping_ur_df.groupby('OrfIDsOverlapping').agg(
            {'genomic_UR': 'first',
             'ORFs_sharing_region': 'first',
             'OrfPosition': 'first',
             'ID': lambda x: ','.join(x),
             'OrfTransID': lambda x: ','.join(x),
             'OrfPositionsOverlapping': 'first',
             'OrfIDsOverlapping': 'first',
             'OverlapPercentage': 'max',
             'geneID': 'first',
             }).reset_index(drop=True))

    gene_dfs = {gene: gene_df.reset_index(drop=True).copy(
    ) for gene, gene_df in dna_distinct_ur_df.groupby(id)}
    gene_dfs = {gene: summarize_overlapping_urs(
        gene_df, agg_col_index) for gene, gene_df in gene_dfs.items()}
    dna_distinct_ur_df = copy.deepcopy(pd.concat(
        gene_dfs.values()).reset_index(drop=True))

    dna_distinct_ur_df['OrfPosition'].value_counts(
    ).reset_index().to_csv(os.path.join(outdir, f'distinct_URs_per_position_{id}.csv'))
    dna_distinct_ur_df.to_csv(os.path.join(
        outdir, f'dna_distinct_ur_df_{id}.csv'))

    return dna_distinct_ur_df, dna_overlapping_ur_df


In [108]:
    dna_ur_df = load_dna_ur_df(
        ur_path)
    predicted_so_orfs, so_transcripts = load_so_results(so_results)
    all_predicted_so_orfs, predicted_so_orfs = explode_so_df(
        predicted_so_orfs)
    all_predicted_so_orfs = get_so_position_in_transcript(
        all_predicted_so_orfs)
    dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)

    # get completely overlapping URs on gene level
    dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
        lambda x: x.split('|')[0])

In [109]:
    dna_distinct_ur_df_trans_test, dna_overlapping_ur_df_trans_test = overlapping_ur_df_by_id(
        dna_ur_df, outdir, 'OrfTransID', 6)

In [97]:
id = 'OrfTransID'

In [101]:
dna_ur_df

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
0,ORF-10007,100174543,100174682,7,ENSG00000213420|ENST00000471717:ORF-10007:173:...,ENSG00000213420|ENST00000471717,first,7_100174543_100174682,0.0,{first},{ORF-10007},ENSG00000213420
1,ORF-10008,100173810,100173834,7,ENSG00000213420|ENST00000471717:ORF-10008:1622...,ENSG00000213420|ENST00000471717,last,7_100173810_100173834,0.0,{last},{ORF-10008},ENSG00000213420
2,ORF-10062,214334020,214334197,1,ENSG00000143499|ENST00000471645:ORF-10062:4017...,ENSG00000143499|ENST00000471645,last,1_214334020_214334197,0.0,{last},{ORF-10062},ENSG00000143499
3,ORF-10063,214331070,214331219,1,ENSG00000143499|ENST00000471645:ORF-10063:130:...,ENSG00000143499|ENST00000471645,first,1_214331070_214331219,0.0,{first},{ORF-10063},ENSG00000143499
4,ORF-10071,791070,791475,16,ENSG00000127586|ENST00000464728:ORF-10071:1101...,ENSG00000127586|ENST00000464728,middle,16_791070_791475,0.0,{middle},{ORF-10071},ENSG00000127586
...,...,...,...,...,...,...,...,...,...,...,...,...
6387,ORF-9962,130341042,130341127,2,ENSG00000136710|ENST00000465315:ORF-9962:1008:...,ENSG00000136710|ENST00000465315,last,2_130341042_130341127,0.0,{last},{ORF-9962},ENSG00000136710
6388,ORF-9977,45184427,45184450,22,ENSG00000093000|ENST00000469163:ORF-9977:2310:...,ENSG00000093000|ENST00000469163,last,22_45184427_45184450,0.0,{last},{ORF-9977},ENSG00000093000
6389,ORF-9978,45183521,45183570,22,ENSG00000093000|ENST00000469163:ORF-9978:949:1...,ENSG00000093000|ENST00000469163,first,22_45183521_45183570,0.0,{first},{ORF-9978},ENSG00000093000
6390,ORF-9983,184382775,184383010,3,ENSG00000090539|ENST00000470150:ORF-9983:220:1...,ENSG00000090539|ENST00000470150,first,3_184382775_184383010,0.0,{first},{ORF-9983},ENSG00000090539


In [102]:
    # get completely overlapping URs
    gene_dfs = {gene: gene_df.reset_index(drop=True).copy(
    ) for gene, gene_df in dna_ur_df.groupby(id)}
    gene_dfs = {gene: get_max_overlap_of_regions_in_df(
        gene_df, 0.2) for gene, gene_df in gene_dfs.items()}
    dna_overlapping_ur_df = pd.concat(
        gene_dfs.values()).reset_index(drop=True).copy()

    dna_overlapping_ur_df['ORFs_sharing_region'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: len(x))
    dna_overlapping_ur_df['shared_region_type'] = dna_overlapping_ur_df['OrfPositionsOverlapping'].apply(
        lambda x: len(x))
    # frozenset: order within the set does not matter!
    dna_overlapping_ur_df.loc[:, 'OrfIDsOverlapping'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: frozenset(x))

    if id == 'OrfTransID':
        # aggregate ORFs that have overlapping URs with the exact same set of ORFs
        dna_distinct_ur_df = dna_overlapping_ur_df.groupby('OrfIDsOverlapping').agg(
            {'genomic_UR': 'first',
             'ORFs_sharing_region': 'first',
             'OrfPosition': 'first',
             'ID': lambda x: ','.join(x),
             'OrfTransID': 'first',
             'OrfPositionsOverlapping': 'first',
             'OrfIDsOverlapping': 'first',
             'OverlapPercentage': 'max',
             'geneID': 'first',
             }).reset_index(drop=True)
    else:
        # aggregate ORFs that have overlapping URs with the exact same set of ORFs
        dna_distinct_ur_df = dna_overlapping_ur_df.groupby('OrfIDsOverlapping').agg(
            {'genomic_UR': 'first',
             'ORFs_sharing_region': 'first',
             'OrfPosition': 'first',
             'ID': lambda x: ','.join(x),
             'OrfTransID': lambda x: ','.join(x),
             'OrfPositionsOverlapping': 'first',
             'OrfIDsOverlapping': 'first',
             'OverlapPercentage': 'max',
             'geneID': 'first',
             }).reset_index(drop=True)

In [103]:
gene_dfs['ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
0,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last}","{ORF-38893, ORF-38894}",ENSG00000102119
1,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last}","{ORF-38893, ORF-38894}",ENSG00000102119


In [104]:
dna_distinct_ur_df[dna_distinct_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
3807,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [106]:
    gene_dfs = {gene: gene_df.reset_index(drop=True).copy(
    ) for gene, gene_df in dna_distinct_ur_df.groupby(id)}
    gene_dfs = {gene: summarize_overlapping_urs(
        gene_df, 6) for gene, gene_df in gene_dfs.items()}
    dna_distinct_ur_df = pd.concat(
        gene_dfs.values()).reset_index(drop=True).copy()

In [107]:
dna_distinct_ur_df[dna_distinct_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
1274,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [113]:
dna_distinct_ur_df_trans_test[dna_distinct_ur_df_trans_test['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
1274,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [111]:
(dna_distinct_ur_df == dna_distinct_ur_df_trans_test).all()

genomic_UR                 True
ORFs_sharing_region        True
OrfPosition                True
ID                         True
OrfTransID                 True
OrfPositionsOverlapping    True
OrfIDsOverlapping          True
OverlapPercentage          True
geneID                     True
dtype: bool

In [114]:
dna_distinct_ur_df_trans_test[dna_distinct_ur_df_trans_test['OrfTransID'] == 'ENSG00000102119|ENST00000494443']['OrfIDsOverlapping']

1274    frozenset({ORF-38893, ORF-38894})
Name: OrfIDsOverlapping, dtype: object

In [ ]:
# test 2
dna_ur_df = load_dna_ur_df(
    ur_path)
predicted_so_orfs, so_transcripts = load_so_results(so_results)
all_predicted_so_orfs, predicted_so_orfs = explode_so_df(
    predicted_so_orfs)
all_predicted_so_orfs = get_so_position_in_transcript(
    all_predicted_so_orfs)
dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)


# get completely overlapping URs on gene level
dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
    lambda x: x.split('|')[0])

dna_ur_df_unaltered = copy.deepcopy(dna_ur_df)


dna_distinct_ur_df, dna_overlapping_ur_df = overlapping_ur_df_by_id(
    dna_ur_df, outdir, 'geneID', 6)
dna_distinct_ur_df_trans_test2, dna_overlapping_ur_df_trans_test2 = overlapping_ur_df_by_id(
    dna_ur_df_unaltered, outdir, 'OrfTransID', 6)


In [161]:
dna_ur_df = load_dna_ur_df(
    ur_path)
predicted_so_orfs, so_transcripts = load_so_results(so_results)
all_predicted_so_orfs, predicted_so_orfs = explode_so_df(
    predicted_so_orfs)
all_predicted_so_orfs = get_so_position_in_transcript(
    all_predicted_so_orfs)
dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)
dna_ur_df[dna_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']
# get completely overlapping URs on gene level
dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
    lambda x: x.split('|')[0])

dna_ur_df_unaltered = copy.deepcopy(dna_ur_df)

In [162]:
dna_ur_df_unaltered[dna_ur_df_unaltered['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
2061,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,0.0,{first},{ORF-38893},ENSG00000102119
2062,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,0.0,{last},{ORF-38894},ENSG00000102119


In [163]:
dna_distinct_ur_df, dna_overlapping_ur_df = overlapping_ur_df_by_id(
    dna_ur_df, outdir, 'geneID', 6)

In [164]:
dna_ur_df_unaltered[dna_ur_df_unaltered['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
2061,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,0.0,"{first, last, middle}","{ORF-18533, ORF-18532, ORF-38893, ORF-38894}",ENSG00000102119
2062,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,0.0,"{first, last, middle}","{ORF-18533, ORF-18532, ORF-38893, ORF-38894}",ENSG00000102119


In [156]:
id(dna_ur_df_unaltered['ID'].iloc[0])

TypeError: 'str' object is not callable

In [154]:
dna_distinct_ur_df_trans_test2[dna_distinct_ur_df_trans_test2['OrfTransID'] == 'ENSG00000102119|ENST00000494443']['OrfIDsOverlapping']

Series([], Name: OrfIDsOverlapping, dtype: object)

In [123]:
    so_categorization_df, all_predicted_so_orfs = so_transcript_categorization(
        dna_overlapping_ur_df_trans_test2, all_predicted_so_orfs)

    so_categorization_df = format_categorization_df(so_categorization_df)

In [124]:
so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000494443']

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,URInLastORF,URInMiddleORF
1390,ENST00000494443,"ORF-38894,ORF-38893","265,57",ENSG00000102119,"last,first","X_154380020_154380232,X_154380020_154380096","True,True",2,2,1,1,0


In [125]:
    so_categorization_df = get_overlapping_info(
        dna_distinct_ur_df, so_categorization_df, dna_distinct_ur_df_trans_test2)

UR test True


In [126]:
so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000494443']

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,...,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans,DistinctURInSecondORF,hasDistinctUR,IDfirstORF,IDOverlapfirstORF,OrfsWithDistinctURTrans
1390,ENST00000494443,"ORF-38894,ORF-38893","265,57",ENSG00000102119,"last,first","X_154380020_154380232,X_154380020_154380096","True,True",2,2,1,...,0,"[frozenset({ORF-38893, ORF-38894})]",,1,1,0,"False,True",ORF-38893,[ORF-38894],ORF-38893


In [147]:
    dna_ur_df = load_dna_ur_df(
        ur_path)
    predicted_so_orfs, so_transcripts = load_so_results(so_results)
    all_predicted_so_orfs, predicted_so_orfs = explode_so_df(
        predicted_so_orfs)
    all_predicted_so_orfs = get_so_position_in_transcript(
        all_predicted_so_orfs)
    dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)

    # get completely overlapping URs on gene level
    dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
        lambda x: x.split('|')[0])

    dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)

    # get completely overlapping URs on gene level
    dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
        lambda x: x.split('|')[0])

    dna_ur_df_unaltered = copy.deepcopy(dna_ur_df)

    dna_distinct_ur_df_trans, dna_overlapping_ur_df_trans = overlapping_ur_df_by_id(
        dna_ur_df_unaltered, outdir, 'OrfTransID', 6)
    dna_distinct_ur_df, dna_overlapping_ur_df = overlapping_ur_df_by_id(
        dna_ur_df, outdir, 'geneID', 6)


    so_categorization_df, all_predicted_so_orfs = so_transcript_categorization(
        dna_overlapping_ur_df_trans, all_predicted_so_orfs)

In [148]:
dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfTransID'] == 'ENSG00000102119|ENST00000494443']['OrfIDsOverlapping']

1274    frozenset({ORF-38893, ORF-38894})
Name: OrfIDsOverlapping, dtype: object

In [88]:
dna_overlapping_ur_df_trans[dna_overlapping_ur_df_trans['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID,ORFs_sharing_region,shared_region_type
1338,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3
1339,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3


In [82]:
dna_ur_df[dna_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
2061,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,0.0,"{first, last, middle}","{ORF-18533, ORF-18532, ORF-38893, ORF-38894}",ENSG00000102119
2062,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,0.0,"{first, last, middle}","{ORF-18533, ORF-18532, ORF-38893, ORF-38894}",ENSG00000102119


In [ ]:
    dna_ur_df = classify_ur_per_orf_position(dna_ur_df, all_predicted_so_orfs)

    # get completely overlapping URs on gene level
    dna_ur_df['geneID'] = dna_ur_df['OrfTransID'].apply(
        lambda x: x.split('|')[0])

In [36]:
    # get completely overlapping URs
    gene_dfs = {gene: gene_df.reset_index(drop=True).copy(
    ) for gene, gene_df in dna_ur_df.groupby('OrfTransID')}
    gene_dfs = {gene: get_max_overlap_of_regions_in_df(
        gene_df, 0.2) for gene, gene_df in gene_dfs.items()}
    dna_overlapping_ur_df = pd.concat(
        gene_dfs.values()).reset_index(drop=True).copy()

In [40]:
dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID
1338,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last}","{ORF-38893, ORF-38894}",ENSG00000102119
1339,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last}","{ORF-38893, ORF-38894}",ENSG00000102119


In [41]:
    dna_overlapping_ur_df['ORFs_sharing_region'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: len(x))
    dna_overlapping_ur_df['shared_region_type'] = dna_overlapping_ur_df['OrfPositionsOverlapping'].apply(
        lambda x: len(x))
    # frozenset: order within the set does not matter!
    dna_overlapping_ur_df.loc[:, 'OrfIDsOverlapping'] = dna_overlapping_ur_df['OrfIDsOverlapping'].apply(
        lambda x: frozenset(x))

In [42]:
dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID,ORFs_sharing_region,shared_region_type
1338,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last}","frozenset({ORF-38893, ORF-38894})",ENSG00000102119,2,2
1339,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last}","frozenset({ORF-38893, ORF-38894})",ENSG00000102119,2,2


In [43]:
        # aggregate ORFs that have overlapping URs with the exact same set of ORFs
        dna_distinct_ur_df = dna_overlapping_ur_df.groupby('OrfIDsOverlapping').agg(
            {'genomic_UR': 'first',
             'ORFs_sharing_region': 'first',
             'OrfPosition': 'first',
             'ID': lambda x: ','.join(x),
             'OrfTransID': 'first',
             'OrfPositionsOverlapping': 'first',
             'OrfIDsOverlapping': 'first',
             'OverlapPercentage': 'max',
             'geneID': 'first',
             }).reset_index(drop=True)

In [46]:
dna_distinct_ur_df[dna_distinct_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
3807,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [50]:
    gene_dfs = {gene: gene_df.reset_index(drop=True).copy(
    ) for gene, gene_df in dna_distinct_ur_df.groupby('OrfTransID')}

In [53]:
gene_dfs['ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
0,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [ ]:
# example of 2 non-voerlapping URs
gene_dfs['ENSG00000064687|ENST00000433129']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
0,19_1042661_1042744,1,middle,ENSG00000064687|ENST00000433129:ORF-1936:1094:...,ENSG00000064687|ENST00000433129,{middle},frozenset({ORF-1936}),0.0,ENSG00000064687
1,19_1042398_1042525,1,first,ENSG00000064687|ENST00000433129:ORF-1932:337:9...,ENSG00000064687|ENST00000433129,{first},frozenset({ORF-1932}),0.0,ENSG00000064687


In [55]:
gene_df = gene_dfs['ENSG00000102119|ENST00000494443']

In [56]:
len(gene_df.index)

1

In [ ]:
    if len(gene_df.index) > 1:
        gene_df_return = gene_df.copy()
        # search for pairwise overlaps of the OrfIDsOverlapping
        for index1 in gene_df.index:
            # compare 0-1, 0-2, 0-3, 1-2, 1-3, 2-3
            index2 = index1
            while index2 < len(gene_df.index) - 1:
                index2 = index2 + 1
                orf_id_overlap_1 = gene_df.iloc[index1, col_index]
                orf_id_overlap_2 = gene_df.iloc[index2, col_index]
                # if ORF IDs do overlap
                if len(orf_id_overlap_1 & orf_id_overlap_2) > 0:
                    # check if index still exists or is already removed
                    if index2 in gene_df_return.index:
                        # always keep index1: ensure that one region of the overlapping
                        # ones is kept in the end!
                        gene_df_return = gene_df_return.drop(index=index2)
        return gene_df_return
    else:
        # return gene df if not several URs per gene
        return gene_df

In [57]:
    gene_dfs = {gene: summarize_overlapping_urs(
        gene_df, 6) for gene, gene_df in gene_dfs.items()}

In [58]:
gene_dfs['ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
0,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [59]:
gene_dfs['ENSG00000064687|ENST00000433129']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
0,19_1042661_1042744,1,middle,ENSG00000064687|ENST00000433129:ORF-1936:1094:...,ENSG00000064687|ENST00000433129,{middle},frozenset({ORF-1936}),0.0,ENSG00000064687
1,19_1042398_1042525,1,first,ENSG00000064687|ENST00000433129:ORF-1932:337:9...,ENSG00000064687|ENST00000433129,{first},frozenset({ORF-1932}),0.0,ENSG00000064687


In [60]:
    dna_distinct_ur_df = pd.concat(
        gene_dfs.values()).reset_index(drop=True).copy()

In [62]:
dna_distinct_ur_df[dna_distinct_ur_df['OrfTransID'] == 'ENSG00000064687|ENST00000433129']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
410,19_1042661_1042744,1,middle,ENSG00000064687|ENST00000433129:ORF-1936:1094:...,ENSG00000064687|ENST00000433129,{middle},frozenset({ORF-1936}),0.0,ENSG00000064687
411,19_1042398_1042525,1,first,ENSG00000064687|ENST00000433129:ORF-1932:337:9...,ENSG00000064687|ENST00000433129,{first},frozenset({ORF-1932}),0.0,ENSG00000064687


In [63]:
dna_distinct_ur_df[dna_distinct_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
1274,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [65]:
dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID,ORFs_sharing_region,shared_region_type
1338,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last}","frozenset({ORF-38893, ORF-38894})",ENSG00000102119,2,2
1339,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last}","frozenset({ORF-38893, ORF-38894})",ENSG00000102119,2,2


In [66]:
dna_distinct_ur_df_trans = dna_distinct_ur_df.copy()

In [69]:
dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfTransID'] == 'ENSG00000102119|ENST00000494443']['OrfIDsOverlapping']

1274    frozenset({ORF-38893, ORF-38894})
Name: OrfIDsOverlapping, dtype: object

In [70]:
    dna_overlapping_ur_df = dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfIDsOverlapping'].apply(
        lambda x: len(x) > 1)]

In [71]:
dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == 'ENSG00000102119|ENST00000494443']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
1274,X_154380020_154380096,2,first,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,"{first, last}","frozenset({ORF-38893, ORF-38894})",1.0,ENSG00000102119


In [72]:
    dna_distinct_ur_df, so_categorization_df, dna_distinct_ur_df_trans, \
        dna_overlapping_ur_df_trans = identify_overlapping_unique_regions(
            all_predicted_so_orfs, dna_ur_df, outdir)

In [73]:
    so_categorization_df['NrDistinctURs'] = so_categorization_df.apply(
        lambda x: check_for_overlapping_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
    so_categorization_df['UROverlapWithinTrans'] = so_categorization_df.apply(
        lambda x: get_overlapping_ur_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
    so_categorization_df['NrOverlapURsWithinTrans'] = so_categorization_df['nrOrfsWithUR'] - \
        so_categorization_df['NrDistinctURs']

In [74]:
so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000494443']

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,URInLastORF,URInMiddleORF,NrDistinctURs,UROverlapWithinTrans,NrOverlapURsWithinTrans
1390,ENST00000494443,"ORF-38894,ORF-38893","265,57",ENSG00000102119,"last,first","X_154380020_154380232,X_154380020_154380096","True,True",2,2,1,1,0,1,"[frozenset({ORF-38893, ORF-38894})]",1


In [75]:
dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['geneID'] == 'ENSG00000102119']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
800,X_154380020_154380096,4,first,ENSG00000102119|ENST00000486738:ORF-18532:144:...,ENSG00000102119|ENST00000486738,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",1.0,ENSG00000102119


In [76]:
dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfTransID'] == 'ENSG00000102119|ENST00000494443']['OrfIDsOverlapping']

Series([], Name: OrfIDsOverlapping, dtype: object)

In [9]:
    outdir = os.path.dirname(ur_path)
    # ------------------ LOAD DNA UNIQUE REGIONS ------------------ #
    dna_ur_df = load_dna_ur_df(
        ur_path)
    predicted_so_orfs, so_transcripts = load_so_results(so_results)
    all_predicted_so_orfs, predicted_so_orfs = explode_so_df(
        predicted_so_orfs)
    all_predicted_so_orfs = get_so_position_in_transcript(
        all_predicted_so_orfs)
    dna_distinct_ur_df, so_categorization_df, dna_distinct_ur_df_trans, \
        dna_overlapping_ur_df_trans = identify_overlapping_unique_regions(
            all_predicted_so_orfs, dna_ur_df, outdir)
    

In [ ]:

dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['geneID'] == 'ENSG00000102119']

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
800,X_154380020_154380096,4,first,ENSG00000102119|ENST00000486738:ORF-18532:144:...,ENSG00000102119|ENST00000486738,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",1.0,ENSG00000102119


In [33]:
dna_distinct_ur_df_trans

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID
0,1_196677667_196677690,1,first,ENSG00000000971|ENST00000466229:ORF-8643:544:7...,ENSG00000000971|ENST00000466229,{first},frozenset({ORF-8643}),0.0,ENSG00000000971
1,X_65522550_65522604,2,first,ENSG00000001497|ENST00000677154:ORF-52641:83:1...,ENSG00000001497|ENST00000677154,{first},"frozenset({ORF-75037, ORF-52641})",0.0,ENSG00000001497
2,X_65529139_65529209,4,first,ENSG00000001497|ENST00000677986:ORF-82641:91:1...,ENSG00000001497|ENST00000677986,"{first, middle}","frozenset({ORF-85322, ORF-81427, ORF-79450, OR...",0.0,ENSG00000001497
3,X_65524501_65528999,3,middle,ENSG00000001497|ENST00000678074:ORF-85321:1141...,ENSG00000001497|ENST00000678074,"{first, middle}","frozenset({ORF-85321, ORF-84787, ORF-79513})",0.0,ENSG00000001497
4,X_65524264_65524284,6,middle,ENSG00000001497|ENST00000678074:ORF-85323:1703...,ENSG00000001497|ENST00000678074,{middle},"frozenset({ORF-84781, ORF-81420, ORF-79518, OR...",0.0,ENSG00000001497
...,...,...,...,...,...,...,...,...,...
4053,1_235433099_235433170,1,first,ENSG00000284770|ENST00000647233:ORF-59698:116:...,ENSG00000284770|ENST00000647233,{first},frozenset({ORF-59698}),0.0,ENSG00000284770
4054,3_180654677_180654761,1,first,ENSG00000284862|ENST00000650889:ORF-63584:334:...,ENSG00000284862|ENST00000650889,{first},frozenset({ORF-63584}),0.0,ENSG00000284862
4055,3_180654544_180654623,1,last,ENSG00000284862|ENST00000650889:ORF-63585:1240...,ENSG00000284862|ENST00000650889,{last},frozenset({ORF-63585}),0.0,ENSG00000284862
4056,Y_341806_341882,1,first,ENSG00000292327|ENST00000711113:ORF-68474:37:5...,ENSG00000292327|ENST00000711113,{first},frozenset({ORF-68474}),0.0,ENSG00000292327


In [15]:
dna_overlapping_ur_df_trans[dna_overlapping_ur_df_trans['geneID'] == 'ENSG00000102119']

,OrfID,start,stop,chr,ID,OrfTransID,OrfPosition,genomic_UR,OverlapPercentage,OrfPositionsOverlapping,OrfIDsOverlapping,geneID,ORFs_sharing_region,shared_region_type
1336,ORF-18532,154380020,154380096,X,ENSG00000102119|ENST00000486738:ORF-18532:144:...,ENSG00000102119|ENST00000486738,first,X_154380020_154380096,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3
1337,ORF-18533,154380020,154380879,X,ENSG00000102119|ENST00000486738:ORF-18533:352:...,ENSG00000102119|ENST00000486738,middle,X_154380020_154380879,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3
1338,ORF-38893,154380020,154380096,X,ENSG00000102119|ENST00000494443:ORF-38893:57:3...,ENSG00000102119|ENST00000494443,first,X_154380020_154380096,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3
1339,ORF-38894,154380020,154380232,X,ENSG00000102119|ENST00000494443:ORF-38894:265:...,ENSG00000102119|ENST00000494443,last,X_154380020_154380232,1.0,"{first, last, middle}","frozenset({ORF-18533, ORF-18532, ORF-38893, OR...",ENSG00000102119,4,3


In [ ]:
# def get_overlapping_info(dna_distinct_ur_df, so_categorization_df, dna_distinct_ur_df_trans):
#     '''
#     Add information to categorization_df of non-overlapping ORF URs to consider for 
#     ribo-seq coverage in different formats
    # '''
so_categorization_df['UROverlapWithinTrans'] = ''
so_categorization_df['UROverlapGeneral'] = ''
so_categorization_df['NrDistinctURs'] = 0

# subset for overlapping URs only
dna_overlapping_ur_df = dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfIDsOverlapping'].apply(
    lambda x: len(x) > 1)]

so_categorization_df['NrDistinctURs'] = so_categorization_df.apply(
    lambda x: check_for_overlapping_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
so_categorization_df['UROverlapWithinTrans'] = so_categorization_df.apply(
    lambda x: get_overlapping_ur_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
so_categorization_df['NrOverlapURsWithinTrans'] = so_categorization_df['nrOrfsWithUR'] - \
    so_categorization_df['NrDistinctURs']



In [16]:
so_categorization_df['UROverlapWithinTrans'] = ''
so_categorization_df['UROverlapGeneral'] = ''
so_categorization_df['NrDistinctURs'] = 0

# subset for overlapping URs only
dna_overlapping_ur_df = dna_distinct_ur_df_trans[dna_distinct_ur_df_trans['OrfIDsOverlapping'].apply(
    lambda x: len(x) > 1)]

so_categorization_df['NrDistinctURs'] = so_categorization_df.apply(
    lambda x: check_for_overlapping_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
so_categorization_df['UROverlapWithinTrans'] = so_categorization_df.apply(
    lambda x: get_overlapping_ur_orfs_within_trans(x, dna_overlapping_ur_df), axis=1)
so_categorization_df['NrOverlapURsWithinTrans'] = so_categorization_df['nrOrfsWithUR'] - \
    so_categorization_df['NrDistinctURs']

In [26]:
# ORTransID ENST00000494443
row = so_categorization_df.iloc[1390,:]

In [27]:
orfs_with_ur_list = get_orfs_with_ur(row)
orfs_with_ur_list

['ORF-38894', 'ORF-38893']

In [28]:
    trans_id = row['geneID'] + '|' + row['OrfTransID']
    nr_distinct_urs = nr_of_non_overlapping_urs(
        orfs_with_ur_list, dna_overlapping_ur_df, trans_id)
    nr_distinct_urs

2

In [29]:
    orf_set = frozenset(orfs_with_ur_list)
    dna_overlapping_ur_df_sub = dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == trans_id]

In [32]:
dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == trans_id]

,genomic_UR,ORFs_sharing_region,OrfPosition,ID,OrfTransID,OrfPositionsOverlapping,OrfIDsOverlapping,OverlapPercentage,geneID


In [ ]:
def nr_of_non_overlapping_urs(orfs_with_ur_list, dna_overlapping_ur_df, trans_id):
    '''
    get number of distinct UR ORFs
    '''
    # subset the unique df for the same gene
    orf_set = frozenset(orfs_with_ur_list)
    dna_overlapping_ur_df_sub = dna_overlapping_ur_df[dna_overlapping_ur_df['OrfTransID'] == trans_id]
    overlapping_ur_sets = dna_overlapping_ur_df_sub.apply(
        lambda x: x['OrfIDsOverlapping'].intersection(orf_set), axis=1)
    # reutrn all ORFs with UR, unless there are overlapping ORFs:
    # then at least some of the sets are > 1: subtract the number of overlapping ORFs
    # this works because each ORF is only listed once
    return len(orf_set) - sum(overlapping_ur_sets.apply(lambda x: len(x) - 1 if len(x) > 0 else 0))

In [17]:
so_categorization_df[so_categorization_df['geneID'] == 'ENSG00000102119']

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,URInLastORF,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans
1120,ENST00000486738,"ORF-18535,ORF-18533,ORF-18532","896,352,144",ENSG00000102119,"last,middle,first","nan,X_154380020_154380879,X_154380020_154380096","False,True,True",3,2,1,0,1,"[frozenset({ORF-18533, ORF-18532})]",,1,1
1390,ENST00000494443,"ORF-38894,ORF-38893","265,57",ENSG00000102119,"last,first","X_154380020_154380232,X_154380020_154380096","True,True",2,2,1,1,0,None,,2,0


In [ ]:
# check that overlapping URs within transcript numbers correspond
assert sum(so_categorization_df['UROverlapWithinTrans'].apply(
    lambda x: x != None)) == sum(so_categorization_df['NrOverlapURsWithinTrans'] > 0)

# assign this with the number of distinct URs in Second ORFs
so_categorization_df['DistinctURInSecondORF'] = so_categorization_df['NrDistinctURs'] - \
    so_categorization_df['URInFirstORF']

# keep the UR only for one ORF if overlap: always for the more 5' one!
# have it as a True,False list etc
# prepare the different cols accordingly
so_categorization_df['hasDistinctUR'] = so_categorization_df['hasUR'].apply(
    lambda x: [eval(ur_indicator) for ur_indicator in x.split(',')])
so_categorization_df['OrfPosition'] = so_categorization_df['OrfPosition'].apply(
    lambda x: x.split(','))
so_categorization_df['OrfID'] = so_categorization_df['OrfID'].apply(
    lambda x: x.split(','))
so_categorization_df['OrfStart'] = so_categorization_df['OrfStart'].apply(
    lambda x: x.split(','))
so_categorization_df['IDfirstORF'] = so_categorization_df.apply(
    lambda x: x.loc['OrfID'][x['OrfPosition'].index('first')], axis=1)

so_categorization_df['IDOverlapfirstORF'] = so_categorization_df.apply(
    lambda row: orf_id_overlapping_first(row), axis=1)

so_categorization_df['OrfsWithDistinctURTrans'] = so_categorization_df.apply(
    lambda row: orfs_for_which_ur_counts(row), axis=1)
so_categorization_df['hasDistinctUR'] = so_categorization_df.apply(
    lambda row: assign_has_distinct_ur(row), axis=1)

# NR distinct URs == Nr ORFs with UR minus what overlaps and is not counted
# here there is only one set of ORFs that overlaps per trnascript
# this assertion might need to be adapted for other datasets
print('UR test', (so_categorization_df['NrDistinctURs'] == so_categorization_df.apply(
    lambda x: x['nrOrfsWithUR'] if x['UROverlapWithinTrans'] == None else x['nrOrfsWithUR'] - len(x['UROverlapWithinTrans'][0]) + 1, axis=1)).all())

# disntinct UR numbers need to correspond!
assert (so_categorization_df['hasDistinctUR'].apply(lambda x: sum([eval(ur_bool) for ur_bool in x.split(',')])) ==
        so_categorization_df['NrDistinctURs']).all()

# ORFs with distinct UR need to be same number as number of distinct URs, as each ORf is only considered for one UR!
assert (so_categorization_df['OrfsWithDistinctURTrans'].apply(
    lambda x: len(x.split(',')) if isinstance(x, str) else 0) == so_categorization_df['NrDistinctURs']).all()

so_categorization_df = format_categorization_df(so_categorization_df)

In [ ]:
so_categorization_df = get_overlapping_info(
        dna_distinct_ur_df, so_categorization_df, dna_distinct_ur_df_trans)

In [17]:
so_categorization_df

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,URInLastORF,URInMiddleORF
0,ENST00000082468,"ORF-95,ORF-93","1264,297",ENSG00000026950,"last,first","6_26409944_26410005,6_26409915_26409976","True,True",2,2,1,1,0
1,ENST00000216019,"ORF-75,ORF-78","294,5641",ENSG00000100201,"first,last","nan,nan","False,False",2,0,0,0,0
2,ENST00000228345,"ORF-88,ORF-87","966,18",ENSG00000002016,"last,first","nan,12_929973_930050","False,True",2,1,1,0,0
3,ENST00000244302,"ORF-44,ORF-45","31,92",ENSG00000124440,"first,last","nan,nan","False,False",2,0,0,0,0
4,ENST00000251366,"ORF-27,ORF-28","25,1207",ENSG00000103067,"first,last","nan,nan","False,False",2,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5525,ENST00000706899,"ORF-84905,ORF-84903","1490,276",ENSG00000206503,"last,first","nan,6_29943543_29943572","False,True",2,1,1,0,0
5526,ENST00000707142,"ORF-82966,ORF-82965,ORF-82968","2113,126,1805",ENSG00000106086,"last,first,middle","nan,nan,7_30049382_30049441","False,False,True",3,1,0,0,1
5527,ENST00000711111,"ORF-80010,ORF-80011","2196,116",ENSG00000007516,"last,first","nan,16_1344317_1344468","False,True",2,1,1,0,0
5528,ENST00000711113,"ORF-68475,ORF-68474,ORF-68478","1057,37,5146",ENSG00000292327,"middle,first,last","nan,Y_341806_341882,Y_334518_334894","False,True,True",3,2,1,1,0


In [ ]:
    assert (so_categorization_df['NrDistinctURs'] == so_categorization_df.apply(
        lambda x: x['nrOrfsWithUR'] if x['UROverlapWithinTrans'] == None else x['nrOrfsWithUR'] - len(x['UROverlapWithinTrans'][0]), axis=1)).all()

In [15]:
    assert (so_categorization_df['genomic_UR'].apply(lambda x:
        len([ur for ur in x.split(',') if ur != 'nan'])) == so_categorization_df['nrOrfsWithUR']).all()

In [44]:
row['UROverlapWithinTrans'] == None

True

In [80]:
so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,...,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans,DistinctURInSecondORF,hasDistinctUR,IDfirstORF,IDOverlapfirstORF,OrfsWithDistinctURTrans
5141,ENST00000698632,"ORF-77166,ORF-77158,ORF-77165,ORF-77157,ORF-77...","3614,1488,2954,726,128,1148,1730,2166",ENSG00000243649,"last,middle,middle,middle,first,middle,middle,...","6_31951097_31951143,6_31947668_31947911,6_3195...","True,True,True,True,True,False,True,True",8,7,1,...,5,"[frozenset({ORF-77165, ORF-77159})]",,6,1,5,"True,True,False,True,True,False,True,True",ORF-77162,None,"ORF-77166,ORF-77158,ORF-77157,ORF-77162,ORF-77..."


In [81]:
for ur in so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']['genomic_UR'].str.split(','):
    print(ur)

['6_31951097_31951143', '6_31947668_31947911', '6_31950437_31950616', '6_31946906_31947617', '6_31946606_31946677', 'nan', '6_31947910_31947939', '6_31948346_31950616']


In [82]:
for orf in so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']['OrfsWithDistinctURTrans'].str.split(','):
    print(orf)

['ORF-77166', 'ORF-77158', 'ORF-77157', 'ORF-77162', 'ORF-77164', 'ORF-77159']


In [83]:
for orf in so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']['OrfID'].str.split(','):
    print(orf)

['ORF-77166', 'ORF-77158', 'ORF-77165', 'ORF-77157', 'ORF-77162', 'ORF-77163', 'ORF-77164', 'ORF-77159']


In [84]:
for orf in so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']['OrfStart'].str.split(','):
    print(orf)

['3614', '1488', '2954', '726', '128', '1148', '1730', '2166']


In [85]:
for orf in so_categorization_df[so_categorization_df['OrfTransID'] == 'ENST00000698632']['hasDistinctUR'].str.split(','):
    print(orf)

['True', 'True', 'False', 'True', 'True', 'False', 'True', 'True']


In [46]:
so_categorization_df[so_categorization_df['UROverlapWithinTrans'].apply(lambda x:  x != None)]

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,...,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans,DistinctURInSecondORF,hasDistinctUR,IDfirstORF,IDOverlapfirstORF,OrfsWithDistinctURTrans
0,ENST00000082468,"ORF-95,ORF-93","1264,297",ENSG00000026950,"last,first","6_26409944_26410005,6_26409915_26409976","True,True",2,2,1,...,0,"[frozenset({ORF-93, ORF-95})]",,1,1,0,"False,True",ORF-93,[ORF-95],ORF-93
30,ENST00000357375,"ORF-657,ORF-659","130,452",ENSG00000142512,"first,last","19_51416262_51416309,19_51416202_51416309","True,True",2,2,1,...,0,"[frozenset({ORF-659, ORF-657})]",,1,1,0,"True,False",ORF-657,[ORF-659],ORF-657
32,ENST00000357852,"ORF-509,ORF-510","192,131",ENSG00000198089,"last,first","22_31614577_31615045,22_31614861_31615020","True,True",2,2,1,...,0,"[frozenset({ORF-510, ORF-509})]",,1,1,0,"False,True",ORF-510,[ORF-509],ORF-510
71,ENST00000391694,"ORF-1167,ORF-1170","606,95",ENSG00000083838,"last,first","19_58479728_58479928,19_58479728_58479928","True,True",2,2,1,...,0,"[frozenset({ORF-1167, ORF-1170})]",,1,1,0,"False,True",ORF-1170,[ORF-1167],ORF-1170
79,ENST00000394321,"ORF-1621,ORF-1620","80,420",ENSG00000090674,"first,last","19_7527629_7527680,19_7527629_7527863","True,True",2,2,1,...,0,"[frozenset({ORF-1621, ORF-1620})]",,1,1,0,"True,False",ORF-1621,[ORF-1620],ORF-1621
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5141,ENST00000698632,"ORF-77166,ORF-77158,ORF-77165,ORF-77157,ORF-77...","3614,1488,2954,726,128,1148,1730,2166",ENSG00000243649,"last,middle,middle,middle,first,middle,middle,...","6_31951097_31951143,6_31947668_31947911,6_3195...","True,True,True,True,True,False,True,True",8,7,1,...,5,"[frozenset({ORF-77165, ORF-77159})]",,6,1,5,"True,True,False,True,True,True,True,True",ORF-77162,None,"ORF-77166,ORF-77158,ORF-77157,ORF-77162,ORF-77..."
5147,ENST00000698677,"ORF-68369,ORF-68371,ORF-68366","1291,638,1527",ENSG00000112473,"middle,first,last","6_33201986_33202261,nan,6_33202222_33202261","True,False,True",3,2,0,...,1,"[frozenset({ORF-68369, ORF-68366})]",,1,1,1,"True,True,False",ORF-68371,None,"ORF-68369,ORF-68371"
5252,ENST00000699150,"ORF-77604,ORF-77605,ORF-77603","451,704,225",ENSG00000107521,"middle,last,first","10_98429688_98429789,10_98429645_98429789,nan","True,True,False",3,2,0,...,1,"[frozenset({ORF-77604, ORF-77605})]",,1,1,1,"True,False,True",ORF-77603,None,"ORF-77604,ORF-77603"
5346,ENST00000700192,"ORF-79610,ORF-79611","201,379",ENSG00000119042,"first,last","2_199327353_199327373,2_199327369_199327391","True,True",2,2,1,...,0,"[frozenset({ORF-79610, ORF-79611})]",,1,1,0,"True,False",ORF-79610,[ORF-79611],ORF-79610


In [93]:
so_categorization_df['hasDistinctUR']

0             False,True
1            False,False
2             False,True
3            False,False
4            False,False
              ...       
5525          False,True
5526    False,False,True
5527          False,True
5528     False,True,True
5529         False,False
Name: hasDistinctUR, Length: 5530, dtype: str

In [99]:
 so_categorization_df['hasDistinctUR'].apply(lambda x: sum([eval(ur_bool) for ur_bool in x.split(',')])) == so_categorization_df['NrDistinctURs']

0       True
1       True
2       True
3       True
4       True
        ... 
5525    True
5526    True
5527    True
5528    True
5529    True
Length: 5530, dtype: bool

In [100]:
    assert (so_categorization_df['hasDistinctUR'].apply(lambda x: sum([eval(ur_bool) for ur_bool in x.split(',')])) == \
        so_categorization_df['NrDistinctURs']).all()

In [102]:
so_categorization_df['OrfsWithDistinctURTrans']

0                    ORF-93
1                       NaN
2                    ORF-87
3                       NaN
4                       NaN
               ...         
5525              ORF-84903
5526              ORF-82968
5527              ORF-80011
5528    ORF-68474,ORF-68478
5529                    NaN
Name: OrfsWithDistinctURTrans, Length: 5530, dtype: str

In [104]:
    assert (so_categorization_df['OrfsWithDistinctURTrans'].apply(
        lambda x: len(x.split(',')) if isinstance(x, str) else 0) == so_categorization_df['NrDistinctURs']).all()

In [ ]:
# on the gene level there are several transcripts that contain different overlapping ORFs
for orftrid in dna_distinct_ur_df[dna_distinct_ur_df['geneID'] == 'ENSG00000001497']['OrfTransID']:
    print(orftrid)

ENSG00000001497|ENST00000677154,ENSG00000001497|ENST00000679261
ENSG00000001497|ENST00000679116,ENSG00000001497|ENST00000678848,ENSG00000001497|ENST00000678547,ENSG00000001497|ENST00000678705,ENSG00000001497|ENST00000679277,ENSG00000001497|ENST00000678074
ENSG00000001497|ENST00000678848,ENSG00000001497|ENST00000678705,ENSG00000001497|ENST00000677986,ENSG00000001497|ENST00000678074
ENSG00000001497|ENST00000678547,ENSG00000001497|ENST00000679277,ENSG00000001497|ENST00000678074


In [ ]:
# but on the transcript level this is not the case
so_categorization_df[so_categorization_df['UROverlapWithinTrans'].apply(lambda x: len(x) > 1 if x != None else False)]

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,...,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans,DistinctURInSecondORF,hasDistinctUR,IDfirstORF,IDOverlapfirstORF,OrfsWithDistinctURTrans


In [36]:
import numpy as np
so_categorization_df['UROverlapWithinTrans'].isna()

0       False
1        True
2        True
3        True
4        True
        ...  
5525     True
5526     True
5527     True
5528     True
5529     True
Name: UROverlapWithinTrans, Length: 5530, dtype: bool

In [58]:
so_categorization_df[(so_categorization_df['nrOrfsWithUR'] > 2) & (~so_categorization_df['UROverlapWithinTrans'].isna())]

,OrfTransID,OrfID,OrfStart,geneID,OrfPosition,genomic_UR,hasUR,nrOrfs,nrOrfsWithUR,URInFirstORF,...,URInMiddleORF,UROverlapWithinTrans,UROverlapGeneral,NrDistinctURs,NrOverlapURsWithinTrans,DistinctURInSecondORF,hasDistinctUR,IDfirstORF,IDOverlapfirstORF,OrfsWithDistinctURTrans
108,ENST00000422370,"ORF-1727,ORF-1716,ORF-1719,ORF-1715","3341,3048,205,12",ENSG00000104866,"last,middle,middle,first","nan,19_45145264_45145352,19_45142241_45142616,...","False,True,True,True",4,3,1,...,2,"[frozenset({ORF-1719, ORF-1715})]",,2,1,1,"True,True,False,True",ORF-1715,[ORF-1719],"ORF-1727,ORF-1716,ORF-1715"
118,ENST00000429300,"ORF-2107,ORF-2104,ORF-2108","1117,777,1427",ENSG00000100033,"middle,first,last","22_18919599_18919631,22_18923009_18923114,22_1...","True,True,True",3,3,1,...,1,"[frozenset({ORF-2108, ORF-2107})]",,2,1,1,"True,True,False",ORF-2104,None,"ORF-2107,ORF-2104"
196,ENST00000460072,"ORF-5245,ORF-5243,ORF-5244","1738,1542,85",ENSG00000138050,"last,middle,first","2_39757248_39757361,2_39757347_39757367,2_3976...","True,True,True",3,3,1,...,1,"[frozenset({ORF-5245, ORF-5243})]",,2,1,1,"False,True,True",ORF-5244,None,"ORF-5243,ORF-5244"
248,ENST00000461701,"ORF-4063,ORF-4058,ORF-4062,ORF-4057,ORF-4061","1553,615,575,294,71",ENSG00000110046,"last,middle,middle,middle,first","nan,11_64913924_64914080,11_64913949_64914080,...","False,True,True,True,True",5,4,1,...,3,"[frozenset({ORF-4062, ORF-4058})]",,3,1,2,"True,False,True,True,True",ORF-4061,None,"ORF-4063,ORF-4062,ORF-4057,ORF-4061"
302,ENST00000463471,"ORF-8349,ORF-8351,ORF-8347","1681,1700,94",ENSG00000157873,"middle,last,first","1_2559751_2561672,1_2559770_2561524,1_2558468_...","True,True,True",3,3,1,...,1,"[frozenset({ORF-8351, ORF-8349})]",,2,1,1,"True,False,True",ORF-8347,None,"ORF-8349,ORF-8347"
335,ENST00000464097,"ORF-10436,ORF-10437,ORF-10438","1263,136,371",ENSG00000196839,"last,first,middle","20_44620402_44620479,20_44622933_44623005,20_4...","True,True,True",3,3,1,...,1,"[frozenset({ORF-10438, ORF-10437})]",,2,1,1,"True,True,False",ORF-10437,[ORF-10438],"ORF-10436,ORF-10437"
407,ENST00000466134,"ORF-10454,ORF-10451,ORF-10455,ORF-10463","4791,2871,10,4139",ENSG00000141959,"last,middle,first,middle","21_44325733_44325957,21_44316135_44316241,21_4...","True,True,True,True",4,4,1,...,2,"[frozenset({ORF-10463, ORF-10454})]",,3,1,2,"False,True,True,True",ORF-10455,None,"ORF-10451,ORF-10455,ORF-10463"
424,ENST00000466772,"ORF-7166,ORF-7165,ORF-7171,ORF-7169,ORF-7167","172,2160,953,65,1570",ENSG00000241973,"middle,last,middle,first,middle","22_20711174_20711340,22_20709998_20710257,22_2...","True,True,True,False,False",5,3,0,...,2,"[frozenset({ORF-7171, ORF-7166})]",,2,1,2,"True,True,False,True,True",ORF-7169,None,"ORF-7166,ORF-7165,ORF-7169,ORF-7167"
442,ENST00000467294,"ORF-8509,ORF-8507,ORF-8511,ORF-8508,ORF-8510,O...","130,1212,1741,1614,859,2677",ENSG00000114388,"first,middle,middle,middle,middle,last","3_50350436_50350572,3_50349320_50349360,3_5034...","True,True,True,True,True,True",6,6,1,...,4,"[frozenset({ORF-8511, ORF-8508})]",,5,1,4,"True,True,False,True,True,True",ORF-8509,None,"ORF-8509,ORF-8507,ORF-8508,ORF-8510,ORF-8512"
490,ENST00000468778,"ORF-7403,ORF-7406,ORF-7401,ORF-7402","1789,1088,1167,388",ENSG00000005448,"last,middle,middle,first","nan,2_74423385_74423596,2_74423385_74423702,2_...","False,True,True,True",4,3,1,...,2,"[frozenset({ORF-7401, ORF-7406})]",,2,1,1,"True,True,False,True",ORF-7402,None,"ORF-7403,ORF-7406,ORF-7402"


In [72]:
so_categorization_df['hasDistinctUR']     = so_categorization_df['hasUR'].apply(
        lambda x: [eval(ur_indicator) for ur_indicator in x.split(',')])

In [73]:
so_categorization_df['hasDistinctUR']

0               [True, True]
1             [False, False]
2              [False, True]
3             [False, False]
4             [False, False]
                ...         
5525           [False, True]
5526    [False, False, True]
5527           [False, True]
5528     [False, True, True]
5529          [False, False]
Name: hasDistinctUR, Length: 5530, dtype: object

In [74]:
so_categorization_df.iloc[5141,:]

OrfTransID                                                   ENST00000698632
OrfID                      ORF-77166,ORF-77158,ORF-77165,ORF-77157,ORF-77...
OrfStart                               3614,1488,2954,726,128,1148,1730,2166
geneID                                                       ENSG00000243649
OrfPosition                last,middle,middle,middle,first,middle,middle,...
genomic_UR                 6_31951097_31951143,6_31947668_31947911,6_3195...
hasUR                               True,True,True,True,True,False,True,True
nrOrfs                                                                     8
nrOrfsWithUR                                                               7
URInFirstORF                                                               1
URInLastORF                                                                1
URInMiddleORF                                                              5
UROverlapWithinTrans                     [frozenset({ORF-77165, ORF-77159})]

In [75]:
row_issue = so_categorization_df.iloc[5141,:]

In [63]:
row['hasDistinctUR']

nan

In [76]:
            overlapping_orf_sets = row['UROverlapWithinTrans']
            if overlapping_orf_sets != None:
                print(overlapping_orf_sets)
                orf_dict = {}
                more_3_prime_orfs = []
                for overlapping_set in overlapping_orf_sets:
                    print(overlapping_set)
                    for orf in overlapping_set:
                        print(orf)
                        orf_index = row['OrfID'].index(orf)
                        orf_start = row['OrfStart'][orf_index]
                        orf_dict[orf] = int(orf_start)

                    most_5_prime = min(orf_dict, key=orf_dict.get)
                    more_3_prime_orfs.extend(
                        [orf for orf in orf_dict.keys() if orf != most_5_prime])

                orfs_no_distinct_ur = [index for index, orf in enumerate(
                    row['OrfID']) if orf in more_3_prime_orfs]
                has_distinct_ur_list = [
                    x if i not in orfs_no_distinct_ur else False for i, x in enumerate(row['hasDistinctUR'])]